<a href="https://colab.research.google.com/github/LuisManuelCatzoliSoriano/Procesos-Estoc-sticos/blob/main/PActi5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Método de Uniformización para una CMTC

Matriz

$$
 \begin{pmatrix}
0 & 2 & 3 & 0 \\
4 & 0 & 2 & 0 \\
0 & 2 & 0 & 2 \\
1 & 0 & 3 & 0 \\
\end{pmatrix}
$$

Teorema (matriz $P(t)$): La matriz de probabilidad de transición $P(t)=[p_{i,j}(t)]$ está dada por

$P(t)=\sum_{k=0}^∞ e^{-rt}\frac{(rt)^k}{k!}\hat{p}^k$

3. (Ejericio para programar) Este último teorema permite aproximar $P(t)$ usando los primeros $M$ términos de la serie infinita. Se obtienen buenos resultados si se elije

$$M≈máx=\left\{ rt+5\sqrt{rt}, 20 \right\}$$

3.1 Use esta propuesta para calular $P(0.5), P(1)$ y $P(5)$ para la matriz R del ejercicio 1.

Librerias que se utilizaran:

In [6]:
import numpy as np
from math import exp, factorial

Defino la función que nos pedira la matriz y el $t$ como argumentos:

In [20]:
def unifor(R, t):

    r = np.max(np.sum(R, axis=1))

    n = R.shape[0]
    P_hat = np.zeros((n, n))

    for i in range(n):
        ri = np.sum(R[i, :])

        for j in range(n):
            if i == j:
                P_hat[i, j] = 1 - ri/r
            else:
                P_hat[i, j] = R[i, j]/r

    rt = r*t
    M = int(np.ceil(max(rt + 5*np.sqrt(rt), 20)))

    P = np.zeros((n, n))

    for k in range(M + 1):
        coef = exp(-rt)*(rt**k)/factorial(k)
        P += coef*np.linalg.matrix_power(P_hat, k)

    return P

Ahora llamo a la función y defino la matriz $R$

In [21]:
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

print("P(0.5)")
print(np.round(unifor(R, 0.5), 6))

print("\nP(1)")
print(np.round(unifor(R, 1), 6))

print("\nP(5)")
print(np.round(unifor(R, 5), 6))

P(0.5)
[[0.250609 0.216965 0.386657 0.14577 ]
 [0.253135 0.238361 0.374409 0.134095]
 [0.169119 0.193615 0.420301 0.216965]
 [0.158017 0.157445 0.398332 0.286206]]

P(1)
[[0.206151 0.203902 0.39871  0.191236]
 [0.208284 0.205341 0.397899 0.188474]
 [0.196758 0.198379 0.400959 0.203902]
 [0.192046 0.193997 0.401471 0.212484]]

P(5)
[[0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]
 [0.2      0.2      0.399999 0.2     ]]


4. Teorema (Cotas de error para $P(t)$):bPara un $t \ge 0$ fijo, sea

$$
P^M(t)=\left[p_{i,j}^M(t)\right]
=
\sum_{k=0}^{M}
e^{-rt}\frac{(rt)^k}{k!}\hat P^k
$$

entonces

$$
|p_{i,j}(t)-p_{i,j}^M(t)|
\le
\sum_{k=M+1}^{\infty}
e^{-rt}\frac{(rt)^k}{k!}
$$

para todo $1\le i,j\le N$.

4.2 (Ejercicio para programar) Este teorema se puede usar así: Suponga que se desea calcular $P(t)$ con una tolerancia $\epsilon$. Elija $M$ tal que

$$
\sum_{k=M+1}^{\infty}
e^{-rt}\frac{(rt)^k}{k!}
\le
\epsilon
$$

Y se puede implementar de acuerdo al siguiente **algoritmo de uniformización para $P(t)$**:

1. Dados $R,t$, $0<\epsilon<1$.

2. Calcular $r$ usando la igualdad en la definición.

3. Calcular $\hat P$.

4. Inicializar

$$
A=\hat P
$$

$$
B=e^{-rt}I
$$

$$
c=e^{-rt}
$$

$$
sum=c
$$

$$
k=1
$$

5. Mientras

$$
sum < 1-\epsilon
$$

hacer:

$$
c=c\cdot\frac{rt}{k}
$$

$$
B=B+cA
$$

$$
A=A\hat P
$$

$$
sum=sum+c
$$

$$
k=k+1
$$

6. $B$ está a $\epsilon$ de $P(t)$.

Repita el ejercicio 3 aplicando este algoritmo con una tolerancia

$$
\epsilon=0.00001
$$

Indique el valor correspondiente de $M$ en cada caso y compare los resultados.

Defino la función que pide la matriz, el tiempo $t$ y el error como argumentos:

In [26]:
def unifor_error(R, t, eps=1e-5):

    r = np.max(np.sum(R, axis=1))

    n = R.shape[0]
    P_g = np.zeros((n,n))

    for i in range(n):
        ri = np.sum(R[i,:])

        for j in range(n):
            if i == j:
                P_g[i,j] = 1 - ri/r
            else:
                P_g[i,j] = R[i,j]/r

    A = P_g.copy()
    B = exp(-r*t)*np.eye(n)

    c = exp(-r*t)
    suma = c
    k = 1

    while suma < 1 - eps:

        c = c*(r*t)/k

        B = B + c*A

        A = A @ P_g

        suma = suma + c

        k += 1

    M = k - 1

    return B, M, P_g

Se repite lo que hicimos en el ejericio 3:

In [27]:
for t in [0.5, 1, 5]:

    P, M, P_g = unifor_error(R, t)

    print(f"\nt = {t}")
    print("M =", M)
    print(np.round(P,6))


t = 0.5
M = 13
[[0.250608 0.216964 0.386656 0.145769]
 [0.253134 0.23836  0.374408 0.134094]
 [0.169119 0.193614 0.4203   0.216964]
 [0.158017 0.157444 0.39833  0.286205]]

t = 1
M = 19
[[0.20615  0.203901 0.398708 0.191235]
 [0.208283 0.20534  0.397898 0.188474]
 [0.196758 0.198379 0.400957 0.203901]
 [0.192045 0.193996 0.401469 0.212483]]

t = 5
M = 56
[[0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]
 [0.199999 0.199999 0.399997 0.199999]]
